# ZuCo Thought Embedding: ZTE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-iyi/zte/blob/main/notebooks/zte_colab.ipynb)

**Mount → train → run the study suite → explore → back up to Drive.** Train on a powerful Colab Pro GPU, explore the results **inline** (tables, charts, figures), and keep everything **permanently on Drive** so a dropped runtime never loses work — then run inference locally.

- **Platform-adaptable & auto-accelerated.** `--device auto` (the default) picks **CUDA (Colab GPU) → Cloud TPU (torch_xla) → Apple MPS → CPU**. Nothing to configure.
- **Resumable & runtime-loss-proof.** Every long run is `--resume`-safe. Point `OUT_ROOT` at Drive (Sections 6/6b) to persist runs the instant they finish, and call `backup_to_drive()` (Section 4) anytime for a provenance-stamped archive + a browsable mirror.
- **Reproducible.** Each run keeps its exact resolved `config.yaml`; `zte-pack` archives carry a `PROVENANCE.json`/`PROVENANCE.md` (git commit + package versions + per-run metrics) so any result can be reproduced and trusted.
- **Single fixed seed (42)** by default for clean, comparable runs; bump to multiple seeds where you want confidence intervals.
- **No Colab surprises.** Section 2 sets the env vars Colab leaves unset and fixes the working directory / output paths so the CLIs never error on a fresh runtime.

> ZTE requires **Python 3.14** (Colab ships an older Python), so we use [`uv`](https://docs.astral.sh/uv/) to provision it — one cell, no system changes. All ZTE code therefore runs via `!uv run …` (the 3.14 venv); the plain notebook kernel is used only to read result files for the inline exploration in Section 8b.

**Pick a GPU runtime now:** `Runtime → Change runtime type → T4/A100 GPU` (or `TPU`).

## 1 · Set up (uv provisions Python 3.14 + installs ZTE)

In [ ]:
%%bash
pip install -q uv
# Clone the repo only if we're not already inside it.
[ -f pyproject.toml ] || [ -d zte ] || git clone --depth 1 https://github.com/victor-iyi/zte.git
[ -f pyproject.toml ] || cd zte
# Provision Python 3.14 + install torch (CUDA wheel on a Colab GPU) and all extras. Cached across runs.
uv python install 3.14
uv sync --group all


## 2 · Bootstrap the environment

Colab does not set the env vars headless plotting / tokenizers expect, and the CLIs use paths relative to the repo root. This cell sets those vars for every `!uv run …` subprocess and creates the `res/` output directories, so nothing errors later. Idempotent — safe to re-run.

In [ ]:
import os

# Enter the repo in the notebook kernel so relative paths + every `!uv run`
# subprocess resolve. This persists across cells (a %%bash `cd` cannot); the setup cell
# above installed ZTE via %%bash.
if os.path.isdir('zte') and not os.path.isfile('pyproject.toml'):
    os.chdir('zte')

# Set in the notebook kernel so every `!uv run` subprocess inherits them.
# MPLBACKEND is FORCED: Colab sets it to an inline backend that crashes a headless subprocess.
os.environ['MPLBACKEND'] = 'Agg'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('res/.cache/matplotlib'))
# Create res/ dirs + report the resolved root/accelerator via the 3.14 venv.
!uv run python -c "from zte.utils import bootstrap; import json; print(json.dumps(bootstrap(chdir=True, quiet=True), default=str, indent=1))"

## 3 · Confirm the accelerator — and how ZTE adapts to it
`--device auto` (the default in every cell) picks the best backend — **CUDA → Cloud TPU → Apple MPS → CPU** — and ZTE then tunes itself to that hardware **automatically, without touching accuracy**:

- **A100 / H100 (CUDA, Ampere+):** bf16 mixed precision + **TF32** matmuls (a large, free speedup; fp32 master weights keep accuracy). Older CUDA falls back to fp16 + GradScaler.
- **Cloud TPU (v6e etc.):** bf16 (TPUs are bf16-native) + **static-shape padding** so XLA compiles once instead of recompiling per batch — padded positions are masked out, so results are unchanged.
- **Apple Silicon (MPS):** stable fp32 (MPS autocast is still maturing) — fully GPU-accelerated locally, and the flagship now runs end-to-end here (the `pdist` op was replaced with a portable equivalent).
- **CPU:** fp32, single-process loading.

DataLoader workers are auto-picked per backend too. The cell below prints both what was **detected** and exactly what **ZTE will use**. Everything is overridable via `--precision`, `--num-workers`, `--compile`, `--static-shapes` on `zte-run`/`zte-benchmark`.

In [ ]:
%%bash
uv run python - <<'PY'
import json

from zte.device import auto_num_workers, resolve_device
from zte.utils.env import accelerator_info

info = accelerator_info()
spec = resolve_device('auto')  # what ZTE will use with --device auto
plan = {
    'backend': spec.kind,
    'device': spec.name,
    'autocast_dtype': str(spec.autocast_dtype).replace('torch.', '') if spec.autocast_dtype else 'fp32',
    'mixed_precision': spec.use_amp,      # bf16 on Ampere+/TPU, fp16 on older CUDA, off on MPS/CPU
    'pin_memory': spec.supports_pin_memory,
    'dataloader_workers_auto': auto_num_workers(spec, -1),
    'tf32_matmul': spec.kind == 'cuda',   # Ampere+ (A100/H100): free matmul speedup
    'static_shapes': spec.kind == 'xla',  # TPU only: fixed-length padding (accuracy-neutral)
}
print(json.dumps({'detected': info, 'zte_will_use': plan}, indent=2))
PY


### (Optional) Cloud TPU
On a **TPU** runtime, install `torch_xla` so `--device auto` selects it. `torch_xla` must match the installed torch; this is best-effort (GPU is the primary, tested path). Uncomment to try:

In [ ]:
# !uv pip install -q torch_xla   # then re-run the accelerator cell above; it should report a Cloud TPU

## 4 · Get data + set up permanent Drive backup
**A) Synthetic (default, no dataset).** A fabricated ZuCo tree — validates the whole pipeline in minutes. The `--synthetic` cells below use it automatically; skip this cell if that is all you want.

**B) Real ZuCo + permanent backup.** Mount Drive, read the dataset **directly** from `Sharables/ZTE/ZuCo Dataset` (mounting is faster than re-downloading), and set up the backup target. Everything this session produces is saved under a **date-stamped** folder `Sharables/ZTE/{RUN_DATE}/` — so a dropped Colab runtime never loses finished work. The cell defines `backup_to_drive()`, used throughout. Shareable ZTE folder: <https://drive.google.com/drive/folders/13EYW1h6dHD5E4YoEWNsKe6ZBHmMU_kFQ>.

In [ ]:
from google.colab import drive  # type: ignore[import-untyped]

drive.mount('/gdrive')

import datetime
import glob
import json
import os
import pathlib
import shutil
import subprocess

# --- One shareable ZTE folder holds everything (data + every session's outputs) ---
# https://drive.google.com/drive/folders/13EYW1h6dHD5E4YoEWNsKe6ZBHmMU_kFQ
ZTE_DRIVE = '/gdrive/My Drive/Sharables/ZTE'
DATA_DIR = f'{ZTE_DRIVE}/ZuCo Dataset'  # the ZuCo .mat files on your Drive
# To RESUME an interrupted session, set RESUME_DATE to its date (e.g. '2026-07-12'); else None = today.
RESUME_DATE = None
RUN_DATE = (
    RESUME_DATE or datetime.date.today().isoformat()
)  # groups this session's outputs on Drive
DRIVE_DIR = f'{ZTE_DRIVE}/{RUN_DATE}'  # everything this session produces backs up here
for sub in ('', '/experiments', '/archives'):
    os.makedirs(f'{DRIVE_DIR}{sub}', exist_ok=True)
# Expose paths to %%bash cells (which can't read Python vars) as $DATA_DIR / $DRIVE_DIR / ...
os.environ.update(ZTE_DRIVE=ZTE_DRIVE, DATA_DIR=DATA_DIR, DRIVE_DIR=DRIVE_DIR, RUN_DATE=RUN_DATE)

print('data found :', os.path.isdir(DATA_DIR))
print('backups -> :', DRIVE_DIR)
!ls "{DATA_DIR}" | head


def _real_runs(experiments: str = 'res/experiments') -> list[pathlib.Path]:
    """Non-synthetic run dirs — skips --synthetic smoke runs (missing flag = treated as real)."""
    keep = []
    for mf in glob.glob(f'{experiments}/*/manifest.json'):
        try:
            synthetic = json.load(open(mf)).get('synthetic', False)
        except Exception:
            synthetic = False
        if not synthetic:
            keep.append(pathlib.Path(mf).parent)
    return sorted(keep)


def backup_to_drive(note: str | None = None, include_synthetic: bool = False) -> None:
    """Back up REAL (non-synthetic) runs to Drive. Safe to call anytime / repeatedly.

    Smoke / --synthetic runs are skipped by default (pass include_synthetic=True to force); if there
    are no real runs this is a friendly no-op, so nothing pollutes Drive. Produces a restorable,
    provenance-stamped best-only zip under {DRIVE_DIR}/archives/ and a browsable mirror of the real
    runs (reports, figures, 3-D explorers, best.pt) under {DRIVE_DIR}/experiments/. Real benchmarks
    are written straight to Drive by the benchmark cell (Section 7).
    """
    runs = None if include_synthetic else _real_runs()
    if runs is not None and not runs:
        print(
            'No real (non-synthetic) runs to back up — skipping Drive. Pass include_synthetic=True to force.'
        )
        return
    ts = datetime.datetime.now().strftime('%H%M%S')
    arch = f'{DRIVE_DIR}/archives/zte_{RUN_DATE}_{ts}.zip'
    cmd = ['uv', 'run', 'zte-pack', 'zip', '--all', '--best-only', '--out', arch]
    if not include_synthetic:
        cmd.append('--skip-synthetic')
    if note:
        cmd += ['--note', note]
    print('-> provenance zip:', arch)
    subprocess.run(cmd, check=False)
    ignore = shutil.ignore_patterns('cache', 'tb', 'bundle', 'ckpt_epoch*.pt', 'last.pt')
    dst = pathlib.Path(DRIVE_DIR) / 'experiments'
    if runs is None:
        src = pathlib.Path('res/experiments')
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True, ignore=ignore)
    else:
        for r in runs:
            shutil.copytree(r, dst / r.name, dirs_exist_ok=True, ignore=ignore)
    print(f'backed up {"all" if runs is None else len(runs)} run(s) to', DRIVE_DIR)


def snapshot_to_drive(
    note: str | None = None,
    targets: list[str] | None = None,
    move: bool = False,
    include_synthetic: bool = False,
) -> str:
    """Zip the FULL working state to Drive so you can continue LOCALLY without GPU time.

    Captures res/experiments + res/cache + res/benchmark + res/explorer in ONE provenance-stamped
    zip (the dataset cache means a local session never re-prepares data). --synthetic experiment
    runs are excluded by default (include_synthetic=True to keep them). Download the single file,
    then `zte-pack unpack <zip> --dest res` on your machine to keep exploring / training offline.
    """
    ts = datetime.datetime.now().strftime('%H%M%S')
    out = f'{DRIVE_DIR}/archives/zte_snapshot_{RUN_DATE}_{ts}.zip'
    cmd = ['uv', 'run', 'zte-pack', 'snapshot', *(targets or []), '--out', out]
    if not include_synthetic:
        cmd.append('--skip-synthetic')
    if note:
        cmd += ['--note', note]
    if move:
        cmd.append('--move')
    print('-> full snapshot ->', out)
    subprocess.run(cmd, check=False)
    print('snapshot on Drive:', out)
    return out


def remove_from_res(*names: str) -> None:
    """Easily delete run dirs / res/ subpaths locally to free space (does NOT touch Drive).

    remove_from_res('colab_exp6')                  # a run name under res/experiments/
    remove_from_res('res/benchmark', 'res/cache')   # any res/ subpath
    """
    for n in names:
        p = pathlib.Path(n)
        if not p.exists():
            p = pathlib.Path('res/experiments') / n  # bare run name
        if p.exists():
            shutil.rmtree(p)
            print('removed', p)
        else:
            print('not found:', n)


def restore_from_drive(
    run_date: str | None = None, drive_sub: str = 'experiments', local: str = 'res/experiments'
) -> None:
    """Pull a Drive session's runs back to local so you can resume after a runtime reset.

    Copies {ZTE_DRIVE}/<date>/<drive_sub>/* -> <local>/ (checkpoints, config, eval), then re-run the
    training cell with --resume: finished runs skip, interrupted ones continue from their last checkpoint.
    Defaults to this session's RUN_DATE + the flat experiments/ dir; for the LOSO sweep pass
    restore_from_drive(drive_sub='loso', local='res/experiments/loso').
    """
    date = run_date or RUN_DATE
    src = pathlib.Path(f'{ZTE_DRIVE}/{date}/{drive_sub}')
    if not src.is_dir():
        print('nothing to restore at', src)
        return
    dst = pathlib.Path(local)
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(
        f'restored {len(list(src.iterdir()))} item(s) from {src} -> {dst}. Re-run with --resume to continue.'
    )


def mirror_to_drive(local: str, drive_sub: str | None = None) -> None:
    """Copy a local dir to Drive (browsable), minus heavy transient files (cache/tb/bundle/last.pt/epoch ckpts).

    Persists FULL runs (eval, figures, interactive, COMPARE.html) after training locally with DRIVE_BACKUP.
    e.g. mirror_to_drive('res/experiments/loso', 'loso').
    """
    src = pathlib.Path(local)
    if not src.is_dir():
        print('nothing to mirror at', local)
        return
    dst = pathlib.Path(DRIVE_DIR) / (drive_sub or src.name)
    ignore = shutil.ignore_patterns('cache', 'tb', 'bundle', 'ckpt_epoch*.pt', 'last.pt')
    shutil.copytree(src, dst, dirs_exist_ok=True, ignore=ignore)
    print(f'mirrored {src} -> {dst}')

## 5 · Train an experiment — the new SOTA model (MOSAIC)
One full run: prepare → train → evaluate, catalogued under Drive, with the honest **scoreboard** at the top of `report.md`. `experiments/sota_loso.yaml` already includes **electrode spatial encoding** and every MOSAIC lever (`docs/METHODS.md`) — set `CONFIG` to any `experiments/*.yaml` to run a different one, and `HOLDOUT` to the single stranger held out for this run (Section 6 rotates it over the whole cohort).

**Run 5-i (and optionally 5-ii) BEFORE 5-iii** so the meaning target and electrode geometry are real, not fallbacks.

In [ ]:
# 5-i · (Recommended) Provision REAL meaning vectors so distillation carries semantics
# (MOSAIC §2). Without this, the meaning loss falls back to a hash target (mechanism only).
# Restricts GloVe-300 to the ZuCo vocabulary -> a small file at the path sota_loso.yaml expects.
!uv pip install -q gensim

!uv run python scripts/build_meaning_vectors.py \
    --out res/vectors/glove.300d.txt --model glove-wiki-gigaword-300 \
    --vocab-from experiments/sota_loso.yaml --root "{DATA_DIR}"

In [ ]:
# 5-ii · (Recommended) Real electrode geometry for spatial encoding (MOSAIC §7) — auto.
# ZuCo v1/v2 use the 129-channel EGI HydroCel net with standard channel ordering (confirmed), so
# this reproduces the retained 105 scalp electrodes (drops the outer face/neck ring), writes the
# montage, and wires it into the config. Runs directly — no files to edit.
!uv pip install -q mne
!uv run python scripts/export_montage.py --out res/montage_gsn105.csv --zuco105

from zte.config import ZTEConfig

cfg = ZTEConfig.from_yaml('experiments/sota_loso.yaml')
cfg.dataset.montage_csv = 'res/montage_gsn105.csv'
cfg.to_yaml('experiments/sota_loso.yaml')
print(
    'wired montage -> experiments/sota_loso.yaml (dataset.montage_csv =',
    cfg.dataset.montage_csv,
    ')',
)

In [ ]:
# 5-iii · Train the SOTA model as ONE held-out-subject experiment.
CONFIG = 'experiments/sota_loso.yaml'  # any experiments/*.yaml
HOLDOUT = 'ZAB'  # the held-out 'new brain' for this single run

# Synthetic smoke (quick, no data; stays local) — verifies the whole stack runs:
# !uv run zte-run --config {CONFIG} --synthetic --epochs 3 --name colab_smoke --out-root res/experiments

# Real data — writes everything to Drive as it is produced (persists across runtime resets):
!uv run zte-run --config {CONFIG} --root "{DATA_DIR}" \
    --loso-holdout {HOLDOUT} --name sota_loso --out-root "{DRIVE_DIR}/experiments"

# The honest headline (held-out geometry + cross-subject retrieval, every number a lift over raw):
rep = pathlib.Path(f'{DRIVE_DIR}/experiments/sota_loso_lo{HOLDOUT}/evaluation/report.md')
if rep.exists():
    t = rep.read_text()
    print(t[t.find('## Scoreboard') : t.find('## Verdict')])
# Everything above already persists to Drive (out-root). For a provenance-stamped ZIP archive
# of all real runs, use Section 9 (backup_to_drive).

## 6 · Full LOSO on the same config — the “new brain” sweep (resumable)
Once the single run (Section 5) looks good, rotate the held-out subject over the whole cohort with the **same config**, turning one data point into a **trend** (`COMPARE.html`). `FULL_CFG` selects it — defaults to the SOTA config here. Fast local training + a live per-epoch checkpoint mirror to Drive; every run is `--resume`-safe (finished subjects skip). `SMOKE=1` = synthetic dry-run.

In [ ]:
# Fast synthetic dry-run of the whole sweep (CPU, no data). Stays local.
!FULL_CFG=experiments/sota_loso.yaml SMOKE=1 bash scripts/run_loso.sh

# === REAL LOSO sweep on the SOTA config — fast local training + live checkpoint mirror ===
# Uncomment BOTH lines to launch (multi-hour). --resume skips finished subjects.
# !FULL_CFG=experiments/sota_loso.yaml DRIVE_BACKUP="{DRIVE_DIR}/loso" bash scripts/run_loso.sh "{DATA_DIR}"
# mirror_to_drive('res/experiments/loso', 'loso')

# Extras:  SUBJECTS="ZAB ZDM" (restrict the held-out set)  ·  OUT_ROOT="{DRIVE_DIR}/loso" (write straight to Drive)

## 6b · Full experiment suite — the fixed-seed study driver
`scripts/run_suite.sh` runs the whole bias-controlled study set from `docs/EXPERIMENTS.md` (eye-tracking confound, subject-invariance A/B under LOSO, VICReg anti-collapse ablation, objective sweep, representation) at a **single fixed seed (42)**. Every run is `--resume`-safe, so an interrupted suite continues where it stopped.

**Best of both worlds (the default).** Train on the **fast local disk** while `DRIVE_BACKUP` mirrors each run's `best.pt`/`last.pt` to Drive **every epoch** — so trainable progress is never lost if the runtime dies — and then `backup_to_drive()` writes **everything else** (evaluation, figures, interactive HTML, benchmark) to Drive once training completes. You get fast training I/O *and* full persistence. The benchmark writes straight to Drive via `BENCH_ROOT`. Interrupted? Set `RESUME_DATE` (Section 4), `restore_from_drive()`, and re-run: finished runs skip, the interrupted one continues, restored runs re-evaluate (cheap), and the final `backup_to_drive()` re-syncs. The kept-open alternative — `OUT_ROOT=…/experiments` on Drive — writes *everything* to Drive live (simplest resume, no restore) but pays ~20–30 s/epoch on big raw-conformer checkpoints. `SMOKE=1` = fast synthetic dry-run (stays local).

In [ ]:
# Fast synthetic smoke of the ENTIRE suite (seed 42) -> local res/. Stays local (synthetic isn't backed up).
!SMOKE=1 bash scripts/run_suite.sh

# === REAL suite on Colab — DEFAULT: fast local training + live checkpoint mirror, then full backup ===
# Train on the FAST local disk while DRIVE_BACKUP mirrors each run's best/last.pt to Drive EVERY EPOCH
# (so trainable progress is never lost if the runtime dies), then push EVERYTHING — eval, figures,
# interactive, benchmark — to Drive once training completes. --resume (built in) skips finished runs.
# Uncomment BOTH lines to launch the (multi-hour) real run:
# !DRIVE_BACKUP="{DRIVE_DIR}/experiments" BENCH_ROOT="{DRIVE_DIR}/benchmark" bash scripts/run_suite.sh "{DATA_DIR}"
# backup_to_drive(note="full suite (seed 42)")   # <- writes res/experiments/* (eval/figures/interactive) to Drive

# --- Option (kept open): write EVERYTHING straight to Drive as produced (simplest resume, slower ckpt I/O) ---
# !OUT_ROOT="{DRIVE_DIR}/experiments" BENCH_ROOT="{DRIVE_DIR}/benchmark" bash scripts/run_suite.sh "{DATA_DIR}"

# RESUME after a runtime reset (DRIVE_BACKUP flow): set RESUME_DATE (Section 4), then:
#   restore_from_drive()   # pull mirrored checkpoints back to res/experiments
# and re-run the DRIVE_BACKUP command above (--resume skips done, re-evaluates restored runs) + backup_to_drive().
# (With the OUT_ROOT=Drive option, nothing extra is needed — just re-run with the same OUT_ROOT.)

## 6c · Prove each lever — single-variable ablation (`zte-ablate`)
The scoreboard only becomes *proof* when each lever is tested in isolation. `zte-ablate generate` writes a config sweep that changes **exactly one knob**; run both arms, then `zte-ablate diff` reports that knob's contribution to the held-out LOSO north-star — everything else identical. This is the discipline the original reports could only apply to VICReg.

In [ ]:
KNOB = 'objective.meaning_distill_weight'  # any dotted section.field, e.g. model.factored, dataset.normalize

# 1) generate the one-knob sweep (OFF vs ON) from the SOTA config
!uv run zte-ablate generate --config experiments/sota_loso.yaml --knob {KNOB} --values 0,1 --out-dir experiments/ablate

# 2) run both arms on real data -> Drive (swap --root for --synthetic --epochs 3 for a dry-run)
for cfg in sorted(glob.glob('experiments/ablate/*.yaml')):
    !uv run zte-run --config {cfg} --root "{DATA_DIR}" --loso-holdout ZAB --out-root "{DRIVE_DIR}/ablate"

# 3) diff the two scoreboards -> the knob's isolated contribution on the held-out north-star
metrics = sorted(glob.glob(f'{DRIVE_DIR}/ablate/*/evaluation/metrics.json'))
if len(metrics) == 2:
    !uv run zte-ablate diff --knob {KNOB} --baseline {metrics[0]} --variant {metrics[1]}

## 7 · Benchmark objectives on real ZuCo (the flagship sweep)
A reproducible grid — **all four self-supervised objectives (skip-gram / CBOW / masked / CPC) x RoPE x {EEG-only vs +eye-tracking}** at the fixed **seed 42** — trained and fully evaluated on real ZuCo. It answers ZTE's two load-bearing questions in one table: *which objective best encodes thought*, and *how much of any score is the eye-tracking confound* (EEG-only is the honest headline). Results are ranked by `subject_transfer_lift` (does the same word transfer across brains, above chance) and written **straight to Drive** so the multi-hour sweep survives a runtime reset. A one-line **synthetic smoke** is included (commented) for a seconds-long mechanics check.

In [ ]:
import pandas as pd

# Where the benchmark is written. Drive = persisted across runtime resets (needs Section 4 mounted).
BENCH_OUT: str = f'{DRIVE_DIR}/benchmark' if 'DRIVE_DIR' in dir() else 'res/benchmark'

# ===== REAL benchmark on ZuCo — the flagship comparison (runs by default) =======================
# Grid = all 4 self-supervised objectives x RoPE x {EEG-only vs +eye-tracking}, fixed seed 42, on
# real ZuCo (SR + NR). It settles the two questions that decide whether ZTE is a *standard*:
#   (1) which objective best encodes thought?   (2) how much does the eye-tracking confound inflate it?
# 8 models are trained AND fully evaluated (hours on a Colab GPU) and written straight to Drive, so a
# dropped runtime never loses the sweep. Each cell also saves its resolved config.yaml (reproducible).
!uv run zte-benchmark --root "{DATA_DIR}" --tasks SR,NR --objectives skipgram,cbow,masked,cpc \
    --pos-encodings rope --eye-tracking both --seeds 42 --epochs 20 --batch-size 128 --device auto --out "{BENCH_OUT}"

# ----- Fast synthetic smoke instead (seconds; set BENCH_OUT = "res/benchmark" above, then run) ---
# !uv run zte-benchmark --synthetic --objectives skipgram,masked --pos-encodings rope --eye-tracking off --seeds 42 --epochs 3 --out res/benchmark

# ----- Fuller study (optional): widen any axis --------------------------------------------------
# ...  --pos-encodings rope,sinusoidal,alibi    --tasks SR,NR,TSR    --seeds 42,43,44

# ----- Rank the results (CSV is pre-sorted by subject_transfer_lift — the cross-subject north-star)
bench_df = pd.read_csv(f'{BENCH_OUT}/benchmark.csv')
_head = [
    'objective',
    'eye_tracking',
    'sent_retrieval_top1',
    'subject_transfer_lift',
    'eff_rank_ratio',
    'beats_noise',
]
bench_df[[c for c in _head if c in bench_df.columns]]

In [ ]:
# Compare objectives visually: cross-subject transfer & retrieval, EEG-only vs +eye-tracking.
import matplotlib.pyplot as plt

need = {'objective', 'eye_tracking', 'subject_transfer_lift', 'sent_retrieval_top1'}
if need <= set(bench_df.columns):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    for ax, metric in zip(axes, ['subject_transfer_lift', 'sent_retrieval_top1']):
        bench_df.pivot_table(index='objective', columns='eye_tracking', values=metric).plot.bar(
            ax=ax, rot=0
        )
        ax.set(title=metric, xlabel='objective', ylabel=metric)
        ax.grid(axis='y', alpha=0.3)
        ax.legend(title='eye_tracking')
    fig.suptitle('ZuCo benchmark — objective x eye-tracking (seed 42)')
    plt.tight_layout()
    plt.show()
else:
    print('Run the benchmark cell first (or it was a narrower grid).')

# Persist the benchmark to Drive alongside everything else (safe if you mounted Drive).
if 'backup_to_drive' in dir():
    backup_to_drive(note='benchmark: objectives x eye-tracking (seed 42)')

## 8 · Visualise & interact (HTML)
Build the interactive **Thought-Space Explorer** + **Neuron Atlas** for a run, and the **comparison dashboard** across all runs. The dashboard is small enough to render inline; the 5 MB explorers are best downloaded (Section 9) and opened locally for full 3-D interaction.

In [ ]:
!uv run zte-visualize --run res/experiments/exp6 --kind both
!uv run zte-compare --experiments res/experiments --out res/experiments/COMPARE.html
from IPython.display import HTML  # type: ignore[import-untyped]

HTML(filename='res/experiments/COMPARE.html')  # the scorecard + best-run dashboard, inline

## 8b · Explore the training you just ran — tables, charts & images
A quick, self-contained look at your runs **without leaving the notebook** — reads each run's `manifest.json` / `metrics.json` / figures directly (no ZTE import needed, so it works in the plain Colab kernel). You get a **scorecard DataFrame** across all runs, an interactive **run picker** (a Colab dropdown auto-populated from `res/experiments/`) that shows the selected run's **training curves** and key **evaluation figures** inline, a **comparison bar chart**, and a **metrics deep-dive** table for the selected run. (The PCA-by-subject thumbnail in the Section 8 dashboard now embeds itself as a data-URI, so it renders inline on Colab too.)

In [ ]:
# Scorecard: every run's headline metrics in one tidy table (reads manifest.json).
import pandas as pd

rows = []
for mf in sorted(glob.glob('res/experiments/*/manifest.json')):
    m = json.load(open(mf))
    ev = m.get('evaluation') or {}
    verdict = ev.get('verdict') or {}
    passed = sum(1 for v in verdict.values() if v is True) if isinstance(verdict, dict) else None
    rows.append(
        {
            'run': pathlib.Path(mf).parent.name,
            'train_loss': m.get('final_train_loss'),
            'retrieval_top1': ev.get('sentence_retrieval_top1'),
            'subject_transfer_top1': ev.get('subject_transfer_top1'),
            'eff_rank_ratio': ev.get('effective_rank_ratio'),
            'checks_passed': passed,
        }
    )
scorecard = pd.DataFrame(rows).sort_values('retrieval_top1', ascending=False, na_position='last')
scorecard.reset_index(drop=True)

In [ ]:
# Pick a run from a dropdown and view its training + evaluation figures (interactive, Colab widgets).
import glob
import os

from IPython.display import Image, Markdown, display  # type: ignore[import-untyped]

RUNS = sorted(
    os.path.basename(os.path.dirname(p)) for p in glob.glob('res/experiments/*/manifest.json')
)
FIGURES = [
    ('Training curves (loss / lr)', 'checkpoints/training_curves.png'),
    ('PCA of embeddings by subject', 'evaluation/figures/pca_by_subject.png'),
    ('Sentence retrieval (Top-K vs chance)', 'evaluation/figures/retrieval_sentence.png'),
    ('Embedding health (per-dim std + PCA spectrum)', 'evaluation/figures/embedding_health.png'),
    ('Linear-probe comparison', 'evaluation/figures/probe_linear.png'),
]


def show_run_figures(run: str) -> None:
    """Render one run's figures inline (embeds the PNGs, so they show on Colab too)."""
    base = f'res/experiments/{run}'
    display(Markdown(f'### `{run}` — training & evaluation figures'))
    for caption, rel in FIGURES:
        path = f'{base}/{rel}'
        if os.path.exists(path):
            display(Markdown(f'**{caption}** — `{rel}`'))
            display(Image(filename=path))
        else:
            display(Markdown(f'_missing: {rel}_'))


run_selector = None
if not RUNS:
    print('No runs yet — train one first (Section 5), then re-run this cell.')
else:
    try:
        import ipywidgets as widgets  # type: ignore[import-untyped]

        run_selector = widgets.Dropdown(options=RUNS, value=RUNS[0], description='Run:')
        # Reactive: changing the dropdown re-renders below; run_selector.value = current pick.
        widgets.interact(show_run_figures, run=run_selector)
    except ImportError:  # no widgets (e.g. plain terminal) -> just show the first run
        show_run_figures(RUNS[0])

In [ ]:
# Comparison bar chart across runs (matplotlib, from the scorecard).
import matplotlib.pyplot as plt

comp = scorecard.dropna(subset=['retrieval_top1']).set_index('run')
if len(comp):
    ax = comp[['retrieval_top1', 'eff_rank_ratio']].plot.bar(
        figsize=(1.6 * len(comp) + 3, 4), rot=15
    )
    ax.set(title='Runs compared — retrieval Top-1 & effective-rank ratio', ylabel='value')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(title='metric')
    plt.tight_layout()
    plt.show()
else:
    print('No evaluated runs yet — run a training cell first.')

In [ ]:
# Deep-dive: the full metrics.json for the SELECTED run (re-run after changing the dropdown above).
run = run_selector.value if run_selector is not None else (RUNS[0] if RUNS else None)
if run:
    metrics_path = f'res/experiments/{run}/evaluation/metrics.json'
    if os.path.exists(metrics_path):
        metrics = json.load(open(metrics_path))
        flat = pd.json_normalize(metrics, sep='.').T.rename(columns={0: 'value'})
        display(Markdown(f'### `{run}` — full evaluation metrics'))
        display(flat)
    else:
        print('No metrics.json — evaluation may have been skipped for', run)
else:
    print('No run available — train one first (Section 5).')

## 9 · Back up to Drive — lightweight archive + full snapshot
Two complementary saves, both landing under `Sharables/ZTE/{RUN_DATE}/` (permanent & shareable). **Smoke / `--synthetic` runs are skipped by default** — only real runs reach Drive (a session with only smoke runs is a friendly no-op; pass `include_synthetic=True` to force).

- **`backup_to_drive()`** — frequent, cheap. A provenance-stamped **best-only zip** (git commit + versions + each run's `config.yaml` + metrics) *and* a browsable mirror of reports/figures/3-D explorers. Call it after each **real** training step.
- **`snapshot_to_drive()`** — the **continue-locally** bundle. Zips the whole working state — `experiments` (real runs) + `cache` (the dataset cache!) + `benchmark` + `explorer` — into one file. Download it, `zte-pack unpack … --dest res` on your machine, and keep exploring / training **without paying for more GPU time**.

Both carry `PROVENANCE.json`/`PROVENANCE.md`. For long real-data runs, also write straight to Drive via `OUT_ROOT` (Sections 6/6b) so runs persist the instant they finish.

In [ ]:
!uv run zte-pack list

In [ ]:
# Back up EVERYTHING to Drive: all [best] runs as a provenance-stamped zip + a browsable mirror.
backup_to_drive(note=f'session {RUN_DATE}: all best runs')

# What you now have on Drive (permanent, shareable):
#   {DRIVE_DIR}/archives/zte_<date>_<time>.zip   restorable bundle (best.pt + config + eval + PROVENANCE.json/md)
#   {DRIVE_DIR}/experiments/<run>/               browsable reports, figures & 3-D explorers per run
#   {DRIVE_DIR}/benchmark/                        benchmark tables

# Restore later (new Colab session, or your locally) from the newest archive:
# !ls -t "{DRIVE_DIR}/archives"/*.zip | head -1
# !uv run zte-pack unpack "{DRIVE_DIR}/archives/<the>.zip" --dest res/experiments

# Download a single archive to your browser instead:
# from google.colab import files; files.download(f"{DRIVE_DIR}/archives/...zip")

# Free Colab space once it is safely on Drive (add --move to the zip, or):
# !uv run zte-pack clean experiments benchmark --yes

In [ ]:
# FULL snapshot -> Drive: experiments + cache + benchmark + explorer in ONE zip.
# Download this single file and keep exploring / continue training LOCALLY — no GPU needed
# (the dataset cache is bundled, so a local session doesn't re-prepare the data).
snapshot_to_drive(note=f'full working-state snapshot {RUN_DATE}')

# Pick specific subtrees only (e.g. skip the big cache):
# snapshot_to_drive(targets=["experiments", "benchmark", "explorer"])

# Restore locally (or in a fresh Colab) — recreates res/experiments, res/cache, res/benchmark, res/explorer:
# !uv run zte-pack unpack "{DRIVE_DIR}/archives/<the_snapshot>.zip" --dest res

## 10 · Run it locally (inference)
Grab the newest archive from your Drive (`Sharables/ZTE/<date>/archives/`) — it carries `best.pt`, each run's `config.yaml`, the evaluation, and `PROVENANCE.json`/`PROVENANCE.md`. Then, in a terminal on your machine (Apple-silicon MPS is picked up automatically):

```sh
uv sync --group all
uv run zte-pack unpack ~/Downloads/zte_<date>_<time>.zip --dest res/experiments   # or straight from a synced Drive path
cat res/experiments/PROVENANCE.md            # git commit + versions + per-run metrics (how it was produced)

# Re-open the interactive explorer for a run:
open res/experiments/colab_exp6/evaluation/interactive/thought_space_explorer.html

# Extract embeddings from the trained checkpoint (best.pt is enough — shapes + normaliser are baked in):
uv run zte-extract --ckpt res/experiments/colab_exp6/checkpoints/best.pt --root "/path/to/ZuCo Dataset" --out res/embeddings/exp6.npz

# Or re-run just the evaluation / comparison locally:
uv run zte-compare --experiments res/experiments
```

To reproduce a run exactly: `git checkout <commit from PROVENANCE.md>`, `uv sync --group all`, then `uv run zte-run --config res/experiments/<run>/config.yaml --root "/path/to/ZuCo Dataset" --name <run>`.

In [ ]:
# 10b · Encode a brain the model has NEVER seen (zero-shot new subject, MOSAIC §8).
# The encoder takes no subject-ID; identity enters only at the normaliser. So a new person needs
# only a short UNLABELLED baseline (a few hundred words of them reading anything) to compute their
# own scale/covariance -- no labels, no retraining -- then their words embed on the shared frame.
# Here we demonstrate on the held-out subject ZAB (genuinely unseen in the LOSO run above).
from zte.config import ZTEConfig
from zte.data.dataset import ZuCoDataset
from zte.inference.embed import ZTEEmbedder

HOLDOUT = 'ZAB'
run_dir = f'{DRIVE_DIR}/experiments/sota_loso_lo{HOLDOUT}'
emb = ZTEEmbedder.from_checkpoint(f'{run_dir}/checkpoints/best.pt')

# Rebuild the exact feature pipeline the run used, but UN-normalised (the embedder normalises):
dcfg = ZTEConfig.from_yaml(f'{run_dir}/config.yaml').dataset
dcfg.root, dcfg.normalize = DATA_DIR, 'none'
ds = ZuCoDataset(dcfg).build(show_progress=False)
mask = (ds.words['subject'].to_numpy() == HOLDOUT) & ds.presence
feats = ds.features[mask]  # type: ignore[index]  # (n, in_dim) raw features at the model's exact width
baseline_bp, words_bp = feats[:200], feats[200:]  # first ~200 words = the unlabelled baseline

emb.calibrate_subject(baseline_bp, subject_code=HOLDOUT)  # zero-shot calibration
vectors = emb.embed_signals(
    band_power=words_bp,
    subject_codes=[HOLDOUT] * len(words_bp),  # use their own statistics
    show_progress=False,
)
print('encoded', vectors.shape, 'thought vectors for a calibrated new brain')

## 11 · Housekeeping (free space / fresh clone)
Colab disk is small. Remove individual runs with **`remove_from_res('run_name')`** (defined in Section 4), or delete whole `res/` subtrees with `zte-pack clean`; or wipe the checkout and re-clone. In every case your **data and saved runs on Drive are untouched** — only local scratch is removed.

In [ ]:
# Remove specific runs locally (easy; does NOT touch Drive). Frees space after a smoke test:
remove_from_res('colab_exp6', 'colab_exp6_spatial')  # bare run names under res/experiments/
# remove_from_res('res/benchmark', 'res/cache')        # or any res/ subpath

# Or free whole res/ subtrees via the CLI (dry-run without --yes):
# !uv run zte-pack clean experiments cache benchmark --yes
# Wipe everything under res/:  !uv run zte-pack clean all --yes

In [ ]:
# Fresh clone — wipe the checkout and re-clone (e.g. after pushing critical updates).
# Your data + saved runs on Google Drive are NOT touched.
%cd /content
!rm -rf zte
!git clone --depth 1 https://github.com/victor-iyi/zte.git
%cd zte
!uv sync --group all
# Then re-run Section 2 (bootstrap) and re-mount Drive (Section 4).